# Tutorial 09 -- Power Rabi

Sweep the drive amplitude at fixed pulse duration, fit the oscillation, and estimate the `pi`-pulse amplitude.

**Prerequisites.** Tutorials 04 and 06 are recommended first.


## 1. Goal

We will perform a fixed-duration amplitude sweep and fit the resulting oscillation to estimate the drive scale needed for a `pi` pulse.


## 2. Physical Background

At fixed duration, the excited-state population oscillates as the drive area changes. The `pi`-pulse amplitude is the amplitude that produces one half-cycle of the Rabi oscillation.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    ns,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
duration = 40.0 * ns
amplitudes_mhz = np.linspace(0.0, 25.0, 51)
dt = 2.0 * ns


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.2),
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=2,
)
frame = FrameSpec(omega_q_frame=model.omega_q)


## 6. Pulse / Sequence Construction


In [ ]:
responses = []
amplitudes_rad_s = np.array([MHz(value) for value in amplitudes_mhz], dtype=float)
for amplitude in amplitudes_rad_s:
    pulse = Pulse("q", 0.0, duration, square_envelope, amp=float(amplitude), carrier=0.0, label="power_rabi")
    compiled = SequenceCompiler(dt=dt).compile([pulse], t_end=duration + dt)
    result = simulate_sequence(
        model,
        compiled,
        model.basis_state(0, 0),
        {"q": "qubit"},
        config=SimulationConfig(frame=frame, max_step=dt),
    )
    responses.append(final_expectation(result, "P_e"))
responses = np.asarray(responses, dtype=float)


## 7. Running the Simulation


In [ ]:
fit = fit_rabi_vs_amplitude(amplitudes_rad_s, responses, duration=duration)
print(f"Estimated pi amplitude / 2pi = {angular_to_mhz(fit.parameters['pi_amplitude']):.3f} MHz")
print(f"Estimated pi/2 amplitude / 2pi = {angular_to_mhz(fit.parameters['pi_over_two_amplitude']):.3f} MHz")


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.plot(amplitudes_mhz, responses, "o", label="simulation")
ax.plot(amplitudes_mhz, fit.model_y, "-", label="fit")
ax.axvline(angular_to_mhz(fit.parameters["pi_amplitude"]), color="tab:red", linestyle="--", label="pi amplitude")
ax.axvline(angular_to_mhz(fit.parameters["pi_over_two_amplitude"]), color="tab:green", linestyle="--", label="pi/2 amplitude")
ax.set_xlabel("Drive amplitude / 2pi [MHz]")
ax.set_ylabel(r"Final $P_e$")
ax.set_title("Power-Rabi calibration sweep")
ax.legend()
plt.show()


## 9. Physical Interpretation

The fitted `pi` and `pi/2` amplitudes are useful calibration targets for later sequence notebooks. This is also a good place to see the difference between a waveform amplitude and a physical transition frequency: here the sweep variable is the drive strength, not the carrier.


## 10. Exercises / Next Steps

- Repeat the sweep with a Gaussian envelope instead of a square envelope.
- Increase the amplitude range and watch for fit breakdown if the pulse is no longer cleanly two-level.
- Continue to Tutorial 10 for a time-domain Rabi calibration.
